# Requirement 4 · Neurons, multilayer network and backpropagation

McCulloch-Pitts neurons → a learning sigmoid neuron → a multilayer network trained with backpropagation on real
classroom crops, then the same network in **TensorFlow/Keras** and a check that NumPy, PyTorch and TensorFlow
compute identical gradients. See `docs/guide/04_neurons_mlp.md`.

**CPU runtime is enough** (about 10 minutes, most of it extracting ResNet18 features once).
This notebook needs the Kaggle crops from notebook 02 (`data/processed/kaggle_classroom/crops`).

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

In [ ]:
# Colab already has TensorFlow and PyTorch. Recreate the Kaggle crops if this is a fresh runtime:
import os, kagglehub
if not os.path.exists('data/processed/kaggle_classroom/crops'):
    from smartattend.data.kaggle_classroom import prepare
    prepare('data/raw', 'data/processed/kaggle_classroom',
            source=kagglehub.dataset_download('mustafaxgm/classroom-student-engagement-dataset'))

## 1 · Everything in one command (printed walkthrough + training + figures)

In [ ]:
!python -m smartattend neurons

In [ ]:
from IPython.display import Image, display
for name in ['activations_step_sigmoid_tanh', 'neuron_decision_boundaries', 'mlp_crops_training']:
    display(Image(f'reports/figures/nnfund/{name}.png'))

## 2 · Step by step: McCulloch-Pitts and the sigmoid neuron

In [ ]:
from smartattend.nnfund import neural_networks as nn
print('\n'.join(nn.demo_mcculloch_pitts() + nn.demo_sigmoid_neuron()))

## 3 · Forward pass, loss and backward pass by hand

In [ ]:
print('\n'.join(nn.demo_worked_example()))

## 4 · The same network in Keras (TensorFlow)

In [ ]:
import tensorflow as tf
model = tf.keras.Sequential([
    tf.keras.Input(shape=(512,)),
    tf.keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(3),                      # logits for attentive / distracted / inactive
])
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.005, momentum=0.9),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.summary()                                    # 33,027 parameters, same as the NumPy MLP

In [ ]:
print('\n'.join(nn.demo_frameworks()))

## 5 · Tests

In [ ]:
!python -m pytest -q tests/test_neural_networks.py tests/test_mlp_gradients.py tests/test_neurons_optimizers.py